# 07. Event Chain Verification — hash-chained audit

`arcrun` doesn't trust its own logs. Every `Event` an `EventBus` emits is bound to the previous event by a SHA-256 hash, so any tampering — a flipped byte, a reordered turn, a dropped tool result — fails verification at a known index.

Notebook 01 (`arcrun/01-core-react`) showed the surface API. **This is the deep dive.** We'll build chains by hand, drive a real loop with a `MockModel`, mutate events at the first / middle / last positions, and see exactly what `verify_chain()` returns. We'll also compare the arcrun chain (light, in-process) to its siblings in arctrust (Ed25519-signed records) and arcllm (RFC 8785 JCS canonical JSON) — three different tamper-evidence designs for three different surfaces.

**You will learn:**
- How `Event` + `EventBus` produce a chain of hash-linked records
- The exact bytes that go into each `event_hash` (`_canonical_bytes`, `_compute_event_hash`)
- What the genesis sentinel (`GENESIS_PREV_HASH`) is and why it matters
- How to call `verify_chain()` and read every field of `ChainVerificationResult`
- Three classes of tamper detection: self-hash mismatch, chain break, sequence gap
- Why canonical JSON makes hashes deterministic regardless of dict insertion order
- How arcrun's chain compares to `arctrust.audit.SignedChainSink` and `arcllm.trace.JSONLTraceStore`
- The compliance hooks: NIST 800-53 AU-9 / AU-10 — non-repudiation and protection of audit information


## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Pull the public surface of `arcrun.events` into scope. These five names cover the entire chain story: a frozen record (`Event`), an emitter (`EventBus`), a sentinel (`GENESIS_PREV_HASH`), a verifier (`verify_chain`), and the verification result type (`ChainVerificationResult`).

In [ ]:
from arcrun.events import (
    Event,
    EventBus,
    GENESIS_PREV_HASH,
    ChainVerificationResult,
    verify_chain,
)

# Internal helpers — not part of the public surface, but shown in this deep dive
# so we can compute hashes ourselves and compare to what the bus produces.
from arcrun.events import _canonical_bytes, _compute_event_hash

print(
    "public:  ",
    sorted(["Event", "EventBus", "GENESIS_PREV_HASH", "ChainVerificationResult", "verify_chain"]),
)
print("internal:", sorted(["_canonical_bytes", "_compute_event_hash"]))

## 2. Why hash chains in the loop

Every agent action — strategy selection, turn boundaries, tool calls, tool errors, loop completion — is a record. In a federal deployment, those records are evidence: during incident response, you have to be able to *prove* the trail wasn't edited between when it was written and when the auditor read it.

The threat model is concrete. NIST 800-53 calls these out by name:

- **AU-9 (Protection of Audit Information):** unauthorized modification of audit   records must be **detectable**. Storage hardening alone isn't enough — you also   need cryptographic evidence that what you're reading is what was written.
- **AU-10 (Non-Repudiation):** an actor cannot later claim 'I never did that' if the   log says they did, and an attacker cannot reorder or fabricate records to frame   someone else.

A SHA-256 hash chain delivers both. Each event's hash binds the previous event's hash, so editing record *N* invalidates records *N*..end. An attacker would have to rewrite the entire trail from the tampered point forward — and they don't have the future events yet, because the run is live.

What tampering looks like in arcrun:

| Tamper | What changes | What `verify_chain` returns |
|---|---|---|
| Flip a byte in `data` | `event_hash` no longer matches the record's contents | `error='self-hash mismatch'` |
| Drop or reorder events | `prev_hash` of the next event no longer matches | `error='chain break'` |
| Insert / renumber events | `sequence` field no longer equals position in list | `error='sequence gap'` |

All three are tested below.

## 3. The `Event` schema — what's in the record

`Event` is a `@dataclass(frozen=True)` with seven fields. Five carry the payload, two carry the chain. The five payload fields are *all* hashed; the two chain fields are how the hash is *transported*.

| Field | Hashed? | Role |
|---|---|---|
| `type` | yes | event kind, e.g. `'turn.start'`, `'tool.end'`, `'loop.completed'` |
| `timestamp` | yes | `time.time()` at emission |
| `run_id` | yes | binds the event to a specific run |
| `data` | yes | event-specific payload — wrapped in `MappingProxyType` (deep-immutable) |
| `sequence` | yes | monotonic 0-indexed position within the bus |
| `prev_hash` | NO (it's an *input* to the hash) | the previous event's `event_hash` |
| `event_hash` | NO (it's the *output* of the hash) | `SHA-256(prev_hash || canonical_bytes)` |


In [ ]:
import dataclasses

# Direct construction — not normally how you make events; use a bus.
raw = Event(type="demo.start", timestamp=0.0, run_id="r1", data={"k": "v"})
for f in dataclasses.fields(raw):
    print(f"{f.name:<12} = {getattr(raw, f.name)!r}")

Two things to notice:

1. The default for `prev_hash` and `event_hash` is `''` — that's the marker for    *unverified* records. `EventBus.emit()` is the only way to produce events with    real hash values. Calling `verify_chain` on raw-constructed events would fail    immediately (the empty hash won't match SHA-256 of anything).
2. The `data` dict you pass in becomes a `MappingProxyType` automatically (look at    `Event.__post_init__`). This is *deep* immutability — not just frozen-dataclass    immutability of the field, but immutability of the dict's keys and values too.


In [ ]:
from types import MappingProxyType
import pytest

# 1) Frozen dataclass — top-level field assignment forbidden.
with pytest.raises(dataclasses.FrozenInstanceError):
    raw.type = "demo.changed"  # type: ignore[misc]

# 2) data was upgraded to MappingProxyType — even key assignment is forbidden.
assert isinstance(raw.data, MappingProxyType)
with pytest.raises(TypeError):
    raw.data["k"] = "mutated"  # type: ignore[index]

print("Event is deep-immutable: dataclass frozen + MappingProxyType data.")

## 4. Hash construction — `_canonical_bytes` + `_compute_event_hash`

Two private helpers, both intentionally tiny:

```python
def _canonical_bytes(event_type, timestamp, run_id, data, sequence) -> bytes:
    return json.dumps(
        {'type': event_type, 'timestamp': timestamp, 'run_id': run_id,
         'data': dict(data), 'sequence': sequence},
        sort_keys=True,
        separators=(',', ':'),
    ).encode('utf-8')

def _compute_event_hash(prev_hash: str, canonical: bytes) -> str:
    return hashlib.sha256(prev_hash.encode('ascii') + canonical).hexdigest()
```

`sort_keys=True` and `separators=(',', ':')` are the load-bearing arguments. They guarantee that *the same logical event* always serialises to *the exact same bytes*, regardless of how the dict was built. That's how we get to a stable fingerprint.

In [ ]:
import hashlib

# Compute a hash by hand for a single event.
canonical = _canonical_bytes(
    event_type="loop.start",
    timestamp=1700000000.0,
    run_id="walkthrough-1",
    data={"task": "demo"},
    sequence=0,
)
print("canonical bytes:")
print(" ", canonical)
print("first event hash (prev = GENESIS):")
h = _compute_event_hash(GENESIS_PREV_HASH, canonical)
print(" ", h)
print(
    "matches direct hashlib?",
    h == hashlib.sha256(GENESIS_PREV_HASH.encode("ascii") + canonical).hexdigest(),
)

Verify against a real `EventBus`. We seed the bus with a fixed `run_id`, emit one event, then recompute its hash from scratch. The two should match exactly.

In [ ]:
bus = EventBus(run_id="walkthrough-1")
ev = bus.emit("loop.start", {"task": "demo"})

# Recompute from the event's own fields:
expected = _compute_event_hash(
    ev.prev_hash,
    _canonical_bytes(ev.type, ev.timestamp, ev.run_id, ev.data, ev.sequence),
)
print("bus event_hash :", ev.event_hash)
print("recomputed     :", expected)
print("match          :", ev.event_hash == expected)
print("hex digest len :", len(ev.event_hash), "(SHA-256 = 64)")

### `GENESIS_PREV_HASH` — the bottom of the chain

There has to be *something* the first event's hash depends on, or an attacker could splice in a new 'first' event and the chain would still verify. The convention is a fixed sentinel — 64 zero hex digits — that everyone agrees represents 'no prior event'. The first event in any bus has `prev_hash == GENESIS_PREV_HASH`.

In [ ]:
print("GENESIS_PREV_HASH       :", GENESIS_PREV_HASH)
print("len                     :", len(GENESIS_PREV_HASH))
print("all zeros               :", set(GENESIS_PREV_HASH) == {"0"})
print("first emitted ev uses it:", ev.prev_hash == GENESIS_PREV_HASH)

## 5. `EventBus` — emit, observe, the chain that grows

An `EventBus` does three things:

1. **Emit** — assign sequence + prev_hash + event_hash and append to its log.
2. **Observe** — call an optional `on_event(event)` callback. Callback exceptions are    logged and swallowed so a buggy observer can't break the audit chain.
3. **Expose** — `bus.events` returns a *copy* of the log, so callers can iterate    without disturbing internal state.

Internally there's a `threading.Lock` around the emit critical section so the chain stays valid under concurrent emit (verified in `tests/test_events.py::TestThreadSafeEventBus`).

In [ ]:
captured = []


def observer(event):
    captured.append((event.sequence, event.type))


bus = EventBus(run_id="ebus-demo", on_event=observer)
bus.emit("loop.start", {"task": "find primes"})
bus.emit("turn.start", {"turn_number": 1})
bus.emit("tool.start", {"name": "list_primes"})
bus.emit("tool.end", {"name": "list_primes", "ok": True})
bus.emit("turn.end", {"turn_number": 1})
bus.emit("loop.completed", {"status": "success"})

for seq, typ in captured:
    print(f"  seq={seq:<2} type={typ}")
print(f"\n{len(bus.events)} events on the bus, all observed.")

Each event's `prev_hash` is the previous event's `event_hash`. Print the chain to see this directly:

In [ ]:
for ev in bus.events:
    short_prev = ev.prev_hash[:12]
    short_self = ev.event_hash[:12]
    print(f"  seq={ev.sequence}  prev={short_prev}…  self={short_self}…  type={ev.type}")

### Driving the bus from a real loop

The chain isn't just a toy — it's how arcrun records every action of every run. Below we run a tiny loop with an inline `MockModel` (no API key, no httpx, no network) and assert the resulting chain verifies. The events that come out are the real events the React strategy emits in production.

We define `MockModel` *inline* (rather than importing from `tests/conftest.py`) to stay self-contained — and because pytest fixtures don't survive outside a test run.

In [ ]:
from dataclasses import dataclass, field
from typing import Any

from arcrun import run
from arcrun.builtins.task_complete import make_task_complete_tool


@dataclass
class _Usage:
    input_tokens: int = 10
    output_tokens: int = 5
    total_tokens: int = 15


@dataclass
class _LLMResponse:
    content: str | None = None
    tool_calls: list = field(default_factory=list)
    stop_reason: str = "end_turn"
    usage: _Usage = field(default_factory=_Usage)
    cost_usd: float = 0.0001


class MockModel:
    """Inline mock — yields a single end_turn response so the loop terminates fast."""

    def __init__(self, responses: list[_LLMResponse]) -> None:
        self._responses = list(responses)
        self._call_count = 0

    async def invoke(self, messages: list, tools: list | None = None) -> _LLMResponse:
        if self._call_count >= len(self._responses):
            raise RuntimeError("MockModel exhausted")
        resp = self._responses[self._call_count]
        self._call_count += 1
        return resp


model = MockModel([_LLMResponse(content="Done.", stop_reason="end_turn")])

# Jupyter already runs an event loop, so we await directly rather than
# wrapping in asyncio.run(). In a script you'd use asyncio.run(...).
result = await run(
    model=model,
    tools=[make_task_complete_tool()],
    system_prompt="You finish quickly.",
    task="just stop",
    max_turns=2,
)

print(f"{len(result.events)} events emitted from a 1-turn loop:")
for ev in result.events:
    print(f"  seq={ev.sequence}  type={ev.type}")

These came from the *real* React strategy, not from us calling `bus.emit` by hand. The same `verify_chain()` will work on them.

## 6. `verify_chain` — the API

Signature:

```python
def verify_chain(events: list[Event]) -> ChainVerificationResult: ...
```

It walks every event, recomputes the canonical bytes, recomputes the hash, and compares against three invariants:

1. **self-hash** — `event.event_hash == SHA-256(event.prev_hash || canonical(event))`
2. **chain link** — `event.prev_hash == events[i-1].event_hash` (or    `GENESIS_PREV_HASH` for `i == 0`)
3. **sequence** — `event.sequence == i`

The first violation it sees, it stops and returns a result describing what failed. An empty list is valid (vacuously).

In [ ]:
report = verify_chain(result.events)
print("valid              :", report.valid)
print("event_count        :", report.event_count)
print("first_broken_index :", report.first_broken_index)
print("error              :", report.error)
print("type               :", type(report).__name__)

All four fields of `ChainVerificationResult`:

| Field | Type | Meaning |
|---|---|---|
| `valid` | `bool` | True iff every invariant held |
| `event_count` | `int` | total events checked |
| `first_broken_index` | `int \| None` | 0-indexed position of first failure, or None |
| `error` | `str \| None` | one of `'self-hash mismatch'`, `'chain break'`, `'sequence gap'`, or None |


In [ ]:
# Construct results directly to inspect happy / sad shapes
ok = ChainVerificationResult(valid=True, event_count=10)
bad = ChainVerificationResult(
    valid=False,
    event_count=10,
    first_broken_index=4,
    error="self-hash mismatch",
)
print("ok :", ok)
print("bad:", bad)
print("empty chain valid?", verify_chain([]).valid)

## 7. Tamper detection — three positions, three failure modes

We'll build a clean five-event chain, then forge three different attacks. After each we restore the chain to clean for narrative clarity. The pattern is the same in all three:

1. snapshot a copy of the chain
2. mutate one event
3. call `verify_chain` and inspect the result

Because `Event` is frozen + `MappingProxyType`, we can't mutate the existing event object — we use `dataclasses.replace` to construct a *new* event with edited fields, then splice it into the list. This is exactly how an attacker with file-edit access would do it: replace records, hope the chain still validates. It won't.

In [ ]:
from dataclasses import replace
from types import MappingProxyType

victim_bus = EventBus(run_id="tamper-demo")
victim_bus.emit("loop.start", {"task": "compute payroll"})
victim_bus.emit("turn.start", {"turn_number": 1})
victim_bus.emit("tool.start", {"name": "transfer_funds", "amount_usd": 1000})
victim_bus.emit("tool.end", {"name": "transfer_funds", "ok": True})
victim_bus.emit("loop.completed", {"status": "success"})

clean = victim_bus.events  # property returns a fresh list
print("clean chain valid?", verify_chain(clean).valid)
print("\nclean events:")
for ev in clean:
    print(f"  seq={ev.sequence}  type={ev.type}  data={dict(ev.data)}")

### 7.1 Tamper at the **first** event (index 0)

Say the attacker wants to hide that the run started against task `'compute payroll'` and rebrand it as `'send greeting'`. They edit the very first record. Because every subsequent record's hash chain back to this one, the failure is reported at index 0 with `error='self-hash mismatch'`.

In [ ]:
tampered = list(clean)
tampered[0] = replace(tampered[0], data=MappingProxyType({"task": "send greeting"}))

report_first = verify_chain(tampered)
print("valid              :", report_first.valid)
print("first_broken_index :", report_first.first_broken_index)
print("error              :", report_first.error)
print("event_count        :", report_first.event_count)

Index 0, self-hash mismatch — the new event's `data` doesn't hash to the original's stored `event_hash`. Restore for the next demo.

### 7.2 Tamper at the **middle** event (index 2)

More devious: a fraudster wants the `transfer_funds` to look like it was for $10 instead of $1000. They edit the middle record's `data.amount_usd`.

In [ ]:
tampered = list(clean)
tampered[2] = replace(
    tampered[2],
    data=MappingProxyType({"name": "transfer_funds", "amount_usd": 10}),
)

report_mid = verify_chain(tampered)
print("valid              :", report_mid.valid)
print("first_broken_index :", report_mid.first_broken_index)
print("error              :", report_mid.error)

Index 2, self-hash mismatch. Note: even if they also flipped the *next* event's `prev_hash` to point at the forged event's `event_hash`, that next event would itself fail self-hash. The chain is unforgeable without knowledge of every downstream event.

### 7.3 Tamper at the **last** event (index 4)

Final demo: an attacker wants to claim the run completed successfully when actually it failed. They flip `'status'` on the `loop.completed` event.

In [ ]:
tampered = list(clean)
tampered[-1] = replace(
    tampered[-1],
    data=MappingProxyType({"status": "failed"}),
)

report_last = verify_chain(tampered)
print("valid              :", report_last.valid)
print("first_broken_index :", report_last.first_broken_index)
print("error              :", report_last.error)
print("event_count        :", report_last.event_count)

All three positions caught — at the **exact** index of mutation. After all this tampering the original `clean` list is still valid, because `bus.events` returned a *copy*: the tamper edits hit our local list, not the bus's internal log.

In [ ]:
print("clean chain still valid?", verify_chain(clean).valid)
print("bus internal chain     ?", verify_chain(victim_bus.events).valid)

### 7.4 Other failure modes — chain break and sequence gap

The two non-`self-hash` errors are:

- **`chain break`** — `prev_hash` of event *i* doesn't equal `event_hash` of event   *i-1* (or `GENESIS_PREV_HASH` for *i=0*). Triggers when an event is *replaced* with   another well-hashed but unrelated event.
- **`sequence gap`** — `event.sequence != i` after deletion or reordering.

These two often co-fire (deleting an event causes both), but `verify_chain` only reports the *first* it sees while walking the list. We can illustrate `sequence gap` by deleting the middle event:

In [ ]:
shorter = list(clean)
del shorter[2]  # drop the tool.start event
report_drop = verify_chain(shorter)
print("valid              :", report_drop.valid)
print("first_broken_index :", report_drop.first_broken_index)
print("error              :", report_drop.error)
# Either 'chain break' or 'sequence gap' depending on which check fires first
# at the tampered position — both indicate audit-trail tampering.

## 8. Determinism — same logical events, identical hashes

If the canonical bytes were sensitive to dict insertion order, two processes building 'the same' record would compute different hashes and we'd never be able to verify across a serialise/deserialise boundary. `_canonical_bytes` uses `sort_keys=True`, so insertion order doesn't matter.

Here are two passes building the same logical event with different field orders. The canonical bytes — and the resulting `event_hash` for the same `prev_hash` — are byte-identical.

In [ ]:
from collections import OrderedDict

# Pass 1: insertion order a, b, c
data_pass_1 = {"alpha": 1, "beta": 2, "gamma": 3}

# Pass 2: deliberately reversed insertion order, plus OrderedDict for clarity
data_pass_2 = OrderedDict([("gamma", 3), ("beta", 2), ("alpha", 1)])

b1 = _canonical_bytes("demo.deterministic", 1700000000.0, "run-A", data_pass_1, 0)
b2 = _canonical_bytes("demo.deterministic", 1700000000.0, "run-A", data_pass_2, 0)

print("canonical bytes pass 1:", b1)
print("canonical bytes pass 2:", b2)
print("byte-identical?       :", b1 == b2)

h1 = _compute_event_hash(GENESIS_PREV_HASH, b1)
h2 = _compute_event_hash(GENESIS_PREV_HASH, b2)
print("hash pass 1           :", h1)
print("hash pass 2           :", h2)
print("hash-identical?       :", h1 == h2)

Same property holds for nested dicts. Canonical JSON sorts keys at every level — drop a `data` payload with several layers of nesting and the bytes still come out identical.

In [ ]:
nested_a = {"outer": {"z": "last", "a": "first"}, "meta": {"b": 2, "a": 1}}
nested_b = {"meta": {"a": 1, "b": 2}, "outer": {"a": "first", "z": "last"}}

ba = _canonical_bytes("nested", 0.0, "r", nested_a, 0)
bb = _canonical_bytes("nested", 0.0, "r", nested_b, 0)
print("nested canonical match?", ba == bb)
print("  bytes:", ba.decode())

**Why this matters operationally:** when an event is shipped over the network, serialised to JSONL, then re-loaded for verification, the hashes don't drift. Determinism is what makes the chain portable.

## 9. Comparison to sibling chains

Three packages in the Arc stack each have their own tamper-evidence design. They look similar on the surface but solve different problems:

| Package | Symbol | Linkage | Signing | Storage | Use case |
|---|---|---|---|---|---|
| **arcrun** | `EventBus` + `verify_chain` | `event_hash = SHA-256(prev_hash + canonical_json)` | None — relies on storage hardening | In-memory list per run | Per-loop audit during a single agent run |
| **arctrust** | `audit.SignedChainSink` | `event_hash = SHA-256(chain_tip + json(event))` | Ed25519 signature on each `event_hash` | In-memory list of signed records | Cross-run, cross-process audit; survives off-box |
| **arcllm** | `trace.JSONLTraceStore` | RFC 8785 JCS canonical JSON, hash-chained | None at the trace level (signing is the audit sink's job) | Append-only JSONL files, daily rotation | Long-lived trace storage with file rotation and warm-start tamper detection |

Why three separate designs?

- **arcrun** is the inner runtime. The chain has to be *fast* (it's emitted on every   turn) and *light* (it lives next to the loop). No keypair I/O, no disk I/O — pure   in-memory hashing.
- **arctrust** owns the *cross-process* audit. Signed records can leave the runtime   process and be verified by an auditor with the public key — non-repudiation across   trust boundaries.
- **arcllm** owns the *durable trace*. Daily JSONL rotation, RFC 8785 JCS so trace   records survive any JSON library that supports the standard, and tamper detection   on warm start (when a new process picks up an existing trace file).

All three use SHA-256 hash chaining as the *spine*. The variations are in serialization format (vanilla `json.dumps(sort_keys=True)` vs RFC 8785) and what's added on top (Ed25519 signatures, file rotation).

In production you'd typically use **all three together**: arcrun's chain for live loop audit, arctrust's signed chain for evidence that leaves the process, arcllm's trace store for durable storage with rotation. Notebook 04 in `arctrust/` covers `SignedChainSink`; notebook 14 in `arcllm/` covers `JSONLTraceStore`. This notebook stays focused on the in-process arcrun chain.

In [ ]:
# Sanity-check that the three modules are all importable in this kernel.
from arctrust.audit import SignedChainSink
from arcrun.events import EventBus

print("arcrun.EventBus           :", EventBus)
print("arctrust.SignedChainSink  :", SignedChainSink)
try:
    from arcllm.trace.store import JSONLTraceStore

    print("arcllm.JSONLTraceStore    :", JSONLTraceStore)
except ImportError as exc:
    # The exact import path may differ across arcllm versions; this is a comparison
    # cell, not a hard dependency for the rest of the notebook.
    print("arcllm trace store: import path may differ —", exc)

## 10. Compliance use cases

Three concrete uses for the chain in a federal deployment:

### 10.1 Audit reconstruction
After a run completes, the auditor wants to confirm the events they're looking at are exactly the ones that were emitted. They call `verify_chain` on the persisted list and either get `valid=True` (trail intact) or a precise `first_broken_index` telling them where the trail diverged from reality.

### 10.2 Post-incident forensics
Suspect a compromised tool? Pull the events from the run, walk forward from `loop.start`, and check `verify_chain`. If the chain breaks, you know exactly which event was tampered — and because every event includes a `timestamp` and `run_id`, you can correlate against your other observability data.

### 10.3 Sequence reconstruction
Even with a clean chain, the `sequence` field gives you a totally-ordered timeline across concurrent observers — useful when multiple subscribers (telemetry, audit, UI) are looking at the same bus from different threads.

Below is a minimal forensic walk: emit a few events, persist them, simulate the auditor pulling them off disk, verify.

In [ ]:
import json as _json
import tempfile
from pathlib import Path


def _event_to_row(ev: Event) -> dict:
    """Serialize an Event to a JSON-friendly dict.

    We can't use dataclasses.asdict() because MappingProxyType isn't picklable
    (asdict deepcopies). Walking the fields manually is fine — they're all primitive.
    """
    return {
        "type": ev.type,
        "timestamp": ev.timestamp,
        "run_id": ev.run_id,
        "data": dict(ev.data),
        "sequence": ev.sequence,
        "prev_hash": ev.prev_hash,
        "event_hash": ev.event_hash,
    }


# 1) live run emits events
evidence_bus = EventBus(run_id="forensic-1")
evidence_bus.emit("loop.start", {"task": "classified"})
evidence_bus.emit("tool.start", {"name": "redact", "classification": "CUI"})
evidence_bus.emit("tool.end", {"name": "redact", "ok": True})
evidence_bus.emit("loop.completed", {"status": "success"})

# 2) persist as a file the auditor will later read
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "evidence.jsonl"
    with path.open("w") as fh:
        for ev in evidence_bus.events:
            fh.write(_json.dumps(_event_to_row(ev)) + "\n")

    # 3) auditor reads the file later, in a fresh process
    rehydrated: list[Event] = []
    with path.open() as fh:
        for line in fh:
            row = _json.loads(line)
            rehydrated.append(Event(**row))

    auditor_report = verify_chain(rehydrated)
    print("auditor sees", len(rehydrated), "events")
    print("valid?      ", auditor_report.valid)
    print(
        "matches live?",
        auditor_report.valid and rehydrated[-1].event_hash == evidence_bus.events[-1].event_hash,
    )

Determinism comes through here: even after a full round-trip — `Event` -> dict -> `json.dumps` -> file IO -> `json.loads` -> `Event(**row)` — the rehydrated chain still verifies, with the same `event_hash` on the final record. The canonical-bytes contract is what makes this work.

(Implementation note: we walked the fields by hand instead of using `dataclasses.asdict`, because `MappingProxyType` isn't picklable and `asdict` deepcopies its inputs. The serialization is otherwise straightforward.)

## 11. Summary

- **`Event`** is a frozen dataclass with `MappingProxyType` data — deep-immutable.   Five fields participate in the hash; two carry the chain.
- **`_canonical_bytes`** uses `json.dumps(sort_keys=True, separators=(',', ':'))` to   produce deterministic bytes regardless of dict insertion order.
- **`_compute_event_hash`** is `SHA-256(prev_hash || canonical_bytes)` — a single   call, no fancy crypto, no keypair.
- **`GENESIS_PREV_HASH`** is 64 zero hex digits — the sentinel that anchors the first   event so attackers can't splice in a new origin.
- **`EventBus`** emits events with hash chain assigned, observes through an optional   callback (exceptions logged + swallowed), and exposes a *copy* of the events list.   Thread-safe via `threading.Lock`.
- **`verify_chain`** returns a **`ChainVerificationResult`** with four fields:   `valid`, `event_count`, `first_broken_index`, `error`. Errors are one of   `'self-hash mismatch'`, `'chain break'`, `'sequence gap'`.
- **Tamper detection** caught all three positions we tried — first, middle, and last   events. The chain reports the *exact* failing index.
- **Determinism** survives dict reordering and even JSON round-trip, so the chain is   portable across processes and persistence layers.
- **Compared to siblings:** arctrust's `SignedChainSink` adds Ed25519 signing for   cross-process trust, arcllm's `JSONLTraceStore` uses RFC 8785 JCS and adds file   rotation. arcrun stays in-memory and lightweight — the inner-loop audit spine.
- **Compliance:** maps directly to NIST 800-53 AU-9 (Protection of Audit Information)   and AU-10 (Non-Repudiation).

**Public API exercised:** `Event`, `EventBus`, `GENESIS_PREV_HASH`, `ChainVerificationResult`, `verify_chain`, `run`. Internal helpers shown for the deep dive: `_canonical_bytes`, `_compute_event_hash`.

**Next:**
- `arcrun/01-core-react.ipynb` — the surface tour of the loop, where these events   originate.
- `arctrust/04-audit-sinks.ipynb` — `SignedChainSink` for cross-process audit   evidence.
- `arcllm/14-trace-store.ipynb` — `JSONLTraceStore` for durable trace persistence   with daily rotation.